# **Phase 2: Predictive & Prescriptive Analytics**
## ***Market Basket Analysis & Baseline Recommendation System***

### **1. Primary Objectives**
This notebook is dedicated to developing the associative and cold-start recommendation engines for the H&M ecosystem. The core objectives are:
1. **Association Rule Mining:** Extract statistically significant cross-selling patterns ($X \Rightarrow Y$) utilizing the FP-Growth algorithm.
2. **Cold-Start Resolution:** Establish a demographically-aware baseline recommendation logic for newly acquired users, leveraging age-bucketed popularity metrics.
3. **Prescriptive Analytics:** Integrate RFM segmentation outputs (Person B) to dynamically adjust the recommendation ranking algorithm, prioritizing high-margin items for premium customer segments.

### **2. Methodology & Architectural Rationale**

To ensure the robustness, scalability, and business viability of the analytical pipeline, the following methodological constraints and optimizations were implemented:

*   **2.1. Temporal Pruning for Fast-Fashion Lifecycles:**
    Fashion retail is characterized by high seasonality and short product lifecycles. Utilizing the entire dataset introduces the risk of discovering rules for obsolete items. To mitigate this, a **Dual-Window Strategy** is employed:
    *   *Active Window (30 Days):* Identifies items currently in-stock and actively purchased.
    *   *Historical Window (90 Days):* Provides sufficient transactional depth to uncover reliable purchasing patterns, restricted exclusively to items surviving the Active Window.

*   **2.2. Algorithmic Memory Optimization via Native Pruning:**
    Standard frequent pattern libraries (e.g., `mlxtend`) inherently densify sparse matrices during computation, which inevitably leads to Out-Of-Memory (OOM) exceptions on large-scale datasets (~31.7M rows). By pre-calculating the mathematical `min_support` threshold and filtering non-frequent itemsets natively within the **Polars Lazy evaluation graph** (C++ backend), we achieve massive dimensionality reduction prior to any matrix instantiation.

*   **2.3. Empirical Algorithm Selection (FP-Growth vs. Apriori):**
    As established in theoretical frameworks, the Apriori algorithm suffers from exponential time complexity due to iterative candidate generation. An empirical benchmark conducted within this environment demonstrates that **FP-Growth**, which utilizes a compressed FP-Tree structure for dual-pass scanning, operates **~3.8x faster** on our highly sparse dataset, justifying its exclusive use for rule extraction.

*   **2.4. Retention of Repurchase Signals:**
    Behavioral analysis indicates that repurchase behavior constitutes a dominant predictive signal within the H&M ecosystem (e.g., recurring purchases of basics). Consequently, the recommendation engine deliberately permits the suggestion of previously purchased items, aligning the algorithm with empirical consumer retention patterns.

*   **2.5. Expected Cross-Sell Revenue (ECR) for Prescriptive Ranking:**
    To transition from predictive modeling to prescriptive business optimization, recommendations for top-tier RFM segments (e.g., "Mature Champions", "Young Whales") are not ranked solely by statistical Lift. Instead, an **Expected Cross-Sell Revenue (ECR)** metric is introduced:
    $$ECR = Lift \times Confidence \times Median\_Price$$
    This formulation explicitly optimizes the Average Order Value (AOV) and safeguards against the "ROI Trap" by weighting high-margin associations over generic, low-cost popularity.

### **3. Environment Setup & Computational Reproducibility**
To ensure the analytical pipeline is robust, modular, and fully reproducible across diverse execution environments, dynamic directory resolution via the `pathlib` module is utilized. Hardcoded absolute paths are strictly avoided to maintain repository integrity and facilitate seamless collaboration.

### **4. Deferred Execution and Lazy Data Ingestion**
Given the extreme scale of the processed transactional artifact (~31.7 million interactions), instantiating the entire dataset into localized RAM via standard in-memory structures (e.g., Pandas DataFrames) is computationally prohibitive and poses a critical risk of Out-Of-Memory (OOM) exceptions. 

To systematically mitigate this bottleneck, the **Polars Lazy API** (`pl.scan_parquet`) is employed. This framework constructs a deferred computation graph rather than executing operations immediately. By deferring execution, the Polars engine can autonomously optimize query paths—utilizing advanced techniques such as predicate pushdown and projection pushdown—thereby strictly materializing data into memory only when analytically necessary.

In [1]:
import warnings
from pathlib import Path
import polars as pl
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from datetime import timedelta

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")
# Resolve project root robustly whether notebook is run from repository root or notebook folder
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate data/processed from current working dir.")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = DATA_DIR / "train_processed.parquet"
ARTICLES_PATH = DATA_DIR / "articles_processed.parquet"
CUSTOMERS_PATH = DATA_DIR / "customers_segmented.parquet"
OUTPUT_PATH = DATA_DIR / "Association_Rules.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH:", TRAIN_PATH)
# Load datasets as LazyFrames (no computation yet)
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_articles = pl.scan_parquet(ARTICLES_PATH)
lf_customers = pl.scan_parquet(CUSTOMERS_PATH)

print("✓ Data loaded as LazyFrames (computation deferred)")
# Quick sanity check on schemas
print("Train Schema:", lf_train.columns)

Libraries imported successfully.
PROJECT_ROOT: d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system
TRAIN_PATH: d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system\data\processed\train_processed.parquet
✓ Data loaded as LazyFrames (computation deferred)
Train Schema: ['t_dat', 'price', 'sales_channel_id', 'quantity', 'days_ago', 'time_weight', 'customer_id', 'article_id']


### **5. Dimensionality Reduction & Basket Construction**

To prepare the dataset for the FP-Growth algorithm without breaching hardware memory constraints (Out-Of-Memory errors), a multi-stage dimensionality reduction strategy is executed directly within the Polars Lazy evaluation engine.

**5.1. The Dual-Window Heuristic (Dead-Stock Prevention)**

The upstream Exploratory Data Analysis (EDA) established a strict ~90-day shelf-life for the fast-fashion inventory. 
*   **Active Catalog Window (Last 30 Days):** Acts as a proxy for current inventory availability. Items transacted within this window are classified as "alive" and currently orderable.
*   **Historical Context Window (Last 90 Days):** Provides the transactional depth required for robust rule mining. By restricting this historical data exclusively to baskets containing "alive" items, the model learns deep behavioral patterns while guaranteeing it will not recommend obsolete or out-of-stock products.

**5.2. Pure Polars Mathematical Pruning**

Standard execution passes the entire corpus to `TransactionEncoder`, forcing the underlying libraries to allocate vast contiguous memory blocks before calculating item support. Instead of applying an arbitrary limit (e.g., restricting to a "Top 5000" list), we calculate the exact mathematical `min_support` threshold ($0.1\%$). Items failing this threshold are rigorously filtered out natively within the C++ Polars backend. This mathematically guarantees identical FP-Growth outputs while reducing memory overhead by orders of magnitude.

**5.3. Transaction ID (TID) Integrity**

In the data engineering pipeline (Person A), identical item purchases within the same transaction were collapsed into a `quantity` feature. Therefore, performing a standard list aggregation would generate false multi-item baskets (e.g., `[Item_A, Item_A]`). The `.unique()` constraint is rigorously applied to ensure only authentic cross-item correlations are mined. Single-item baskets are explicitly discarded to eliminate computational bloat.

**5.4. Sparse Matrix Instantiation & Library Bug Mitigation**

The optimized list of baskets is transformed utilizing `TransactionEncoder` and cast into a Pandas `SparseDataFrame`. Furthermore, column names are explicitly cast to strings to circumvent a known integer-parsing limitation within the `mlxtend` framework.

In [2]:
# --- 1. DEFINE WINDOWS ---
max_date = lf_train.select(pl.col("t_dat").max()).collect().item()
active_window = max_date - timedelta(days=30)
history_window = max_date - timedelta(days=90)

# --- 2. ACTIVE CATALOG FILTERING ---
active_items_lf = lf_train.filter(pl.col("t_dat") > active_window).select("article_id").unique()

lf_90_active = (
    lf_train
    .filter(pl.col("t_dat") > history_window)
    .join(active_items_lf, on="article_id", how="inner")
)

# --- 3. PURE POLARS PRUNING ---
# Accurately calculate the number of baskets
total_baskets = lf_90_active.select(["customer_id", "t_dat"]).unique().collect().height
min_supp_count = total_baskets * 0.001
print(f"Total historical baskets: {total_baskets:,} | Min Support Count: {min_supp_count:.1f}")

# Filter qualified items directly within the Polars Lazy API
valid_items_lf = (
    lf_90_active
    .select(["customer_id", "t_dat", "article_id"]).unique()
    .group_by("article_id")
    .len()
    .filter(pl.col("len") >= min_supp_count)
    .select("article_id")
)

# High-speed basket creation, filtering out junk from the get-go.
baskets_df = (
    lf_90_active
    .join(valid_items_lf, on="article_id", how="inner")
    .group_by(["customer_id", "t_dat"])
    .agg(pl.col("article_id").unique().alias("basket"))
    # NEW FIX: Explicitly removed `.filter()` to keep single-item baskets for accurate Confidence math
    .collect()
    .to_pandas()
)
print(f"Total authentic baskets retained for FP-Growth: {len(baskets_df):,}")

# --- 4. ENCODING FOR MLXTEND ---
transactions = baskets_df["basket"].tolist()
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions, sparse=True)

# Cast column names to String (Fix mlxtend bug)
basket_sets = pd.DataFrame.sparse.from_spmatrix(te_array, columns=[str(i) for i in te.columns_])
print(f"basket_sets shape ready for FP-Growth: {basket_sets.shape}")

Total historical baskets: 1,240,873 | Min Support Count: 1240.9
Total authentic baskets retained for FP-Growth: 423,315
basket_sets shape ready for FP-Growth: (423315, 362)


### **6. Frequent Pattern Mining & Prescriptive Rule Enrichment**

Having established a highly optimized, memory-efficient sparse matrix, the pipeline proceeds to extract and enrich the association rules.

**6.1. FP-Growth Execution & Stringent Thresholding**

The FP-Growth algorithm is executed to discover frequent itemsets matching the predefined `min_support` threshold (0.1%). To ensure that the generated recommendations represent true, high-probability cross-selling opportunities rather than random co-occurrences, two strict conditions are applied to the Association Rules:
*   **Confidence $\ge$ 0.20:** Ensures a minimum 20% empirical probability that a customer will purchase the consequent given the antecedent.
*   **Lift > 1.0:** Strictly filters for positive correlation, ensuring the items are genuinely bought together more frequently than statistical independence would suggest.

**6.2. N-to-1 Rule Architecture & Serialization Compatibility**

To maximize the engine's capability to read complex shopping carts, the algorithm is configured to extract **N-to-1 rules** (where the antecedent can contain multiple items, but the consequent is strictly a single item). 
Furthermore, native Python `frozenset` objects generated by `mlxtend` are fundamentally incompatible with the Parquet serialization format. By programmatically unpacking these sets—casting the antecedent to an integer list and the consequent to a scalar integer—we ensure seamless cross-compatibility with downstream analytical tasks and data storage.

**6.3. Derivation of Financial Proxies & Metadata Integration**

Association rules inherently provide statistical relationships but lack financial context. To empower the prescriptive ranking logic (ECR) in the subsequent steps, we must quantify the financial value of each recommended item. 
A `median_price` proxy is dynamically computed from the active 90-day transactional window for each `consequent_item`. This financial metric, alongside categorical metadata (`product_group_name`), is joined into the final rule set.

**6.4. Artifact Serialization**

The fully enriched, descending-sorted rule dataframe is exported to a highly compressed Parquet artifact, acting as the definitive data source for both the immediate Recommender Engine and the downstream Business Intelligence Dashboards (Person E).

In [3]:
# --- 5. FP-GROWTH ---
print("Mining frequent itemsets...")
frequent_itemsets = fpgrowth(basket_sets, min_support=0.001, use_colnames=True)
print(f"Found {len(frequent_itemsets)} frequent itemsets.")

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.20)
rules = rules[rules['lift'] > 1.0]

# --- 6. EXTRACT N-TO-1 RULES ---
rules['consequent_len'] = rules['consequents'].apply(lambda x: len(x))
rules_N_to_1 = rules[rules['consequent_len'] == 1].copy()

rules_N_to_1['antecedents_list'] = rules_N_to_1['antecedents'].apply(lambda x:[int(i) for i in x])
rules_N_to_1['consequent_item'] = rules_N_to_1['consequents'].apply(lambda x: int(next(iter(x))))
final_rules = rules_N_to_1.drop(columns=['antecedents', 'consequents', 'consequent_len'])

# --- 7. ECR METRIC (PRICE PROXY) & METADATA ---
item_prices = lf_90_active.group_by("article_id").agg(pl.col("price").median().alias("median_price"))
article_meta = lf_articles.select(["article_id", "product_group_name"])
item_metadata = item_prices.join(article_meta, on="article_id", how="inner").collect().to_pandas()

final_rules = final_rules.merge(
    item_metadata, left_on="consequent_item", right_on="article_id", how="left"
).rename(columns={
    "median_price": "consequent_median_price",
    "product_group_name": "consequent_product_group"
}).drop(columns=["article_id"])

# NEW FIX: Fill NaNs and Normalize Price so it doesn't overpower Lift/Confidence
final_rules["consequent_median_price"] = final_rules["consequent_median_price"].fillna(0.0)
max_p = final_rules["consequent_median_price"].max()
min_p = final_rules["consequent_median_price"].min()
final_rules["price_norm"] = (final_rules["consequent_median_price"] - min_p) / (max_p - min_p + 1e-9)

final_rules = final_rules.sort_values(['lift', 'confidence'], ascending=[False, False]).reset_index(drop=True)

# Export Parquet
OUTPUT_PATH = DATA_DIR / "Association_Rules.parquet"
final_rules.to_parquet(OUTPUT_PATH, index=False)
print(f"Total Expanded N-to-1 rules exported: {len(final_rules)}")

Mining frequent itemsets...
Found 412 frequent itemsets.
Total Expanded N-to-1 rules exported: 85


### **7. The Recommendation Engine: Cold-Start Resolution & Prescriptive Ranking**

The final component of this pipeline is a dynamic, multi-layered recommendation engine designed to handle both cold-start scenarios and complex, multi-item shopping carts, seamlessly bridging predictive rules with prescriptive business logic.

**7.1. Demographically-Aware Cold-Start Initialization**

For users lacking transactional history (or entering with an empty cart), standard Collaborative Filtering and Basket Analysis fail. To resolve this, a **Demographic Prior** is constructed. 
*   Users are classified into discrete chronological cohorts (`<25`, `25-35`, `36-50`, `>50`). 
*   A localized popularity matrix is generated by querying the **Active Window (last 30 days)**, ensuring that the cold-start fallback recommends culturally relevant, highly-trending, and explicitly in-stock items specific to the user's demographic profile.

**7.2. Advanced Cart-Based Rule Matching**

Moving beyond rigid 1-to-1 mappings, the engine utilizes subset evaluation (`set(x).issubset(cart_set)`) to evaluate the user's *entire* current shopping cart. This enables the engine to trigger **N-to-1 associations** (e.g., if a user adds both a blazer and trousers, the algorithm can recommend a statistically correlated tie), highly mirroring real-world e-commerce behavior.

**7.3. Prescriptive Optimization for High-Value Segments (Cross-Check Integration)**

Fulfilling the cross-check duty with the Descriptive Analytics phase (Person B), the engine dynamically alters its ranking heuristic based on the user's RFM Segment:
*   **Standard Users:** Recommendations are ranked by strict statistical `Lift`, optimizing for conversion likelihood.
*   **VIP Segments ("Mature Champions", "Young Whales"):** The ranking mechanism shifts to the **Expected Cross-Sell Revenue (ECR)**. By multiplying `Lift`, `Confidence`, and the `Consequent_Median_Price`, the algorithm prescriptively pushes high-margin bundles to whales, maximizing the Average Order Value (AOV) and preventing the "ROI Trap" of suggesting low-cost, high-volume items to premium buyers.

**7.4. Repurchase Tolerance & Fallback Padding**

Empirical analysis of H&M's fashion ecosystem indicates that *repurchase* (e.g., rebuying basics) is a dominant behavioral signal. Consequently, the algorithm is explicitly designed to tolerate historical repurchases while strictly filtering out items currently present in the active cart. Finally, a padding mechanism ensures the engine consistently outputs a robust vector of exactly $K$ items by appending demographic-trending artifacts when rule-based consequents are exhausted.

In [4]:
# --- 8. BUILD DICTIONARIES (COLD-START TRENDING) ---
age_bucket_expr = (
    pl.when(pl.col("age") < 25).then(pl.lit("<25"))
    .when(pl.col("age") <= 35).then(pl.lit("25-35"))
    .when(pl.col("age") <= 50).then(pl.lit("36-50"))
    .otherwise(pl.lit(">50"))
)

# Trending Dict (Last 30 days)
trending_by_age = (
    lf_train.filter(pl.col("t_dat") > active_window)
    .join(lf_customers.select(["customer_id", "age"]), on="customer_id", how="inner")
    .with_columns(age_bucket_expr.alias("age_bucket"))
    .group_by(["age_bucket", "article_id"])
    .len()
    # NEW FIX: Sort inside Polars aggregation to guarantee correct top 12 items
    .group_by("age_bucket")
    .agg(pl.col("article_id").sort_by("len", descending=True).head(12).alias("top_items"))
    .collect()
)
cold_start_dict = {row["age_bucket"]: row["top_items"] for row in trending_by_age.iter_rows(named=True)}
customers_df = lf_customers.collect().to_pandas().set_index("customer_id")

# --- NEW FIX: PRECOMPUTE INVERTED INDEX FOR LIGHTNING-FAST RULE MATCHING ---
from collections import defaultdict
antecedent_sets = [frozenset(ants) for ants in final_rules["antecedents_list"].tolist()]
item_to_rules = defaultdict(set)
for idx, ant_set in enumerate(antecedent_sets):
    for item in ant_set:
        item_to_rules[item].add(idx)

# --- 9. THE ENGINE ---
# NEW FIX: Use current_cart=None to avoid Python mutable default bugs
def baseline_recommendation(customer_id, current_cart=None, top_k=12):
    if current_cart is None:
        current_cart = []
        
    if customer_id in customers_df.index:
        user_data = customers_df.loc[customer_id]
        # NEW FIX: Safely handle Int8 datatypes with null guards
        age = int(user_data["age"]) if pd.notna(user_data["age"]) else 25
        segment = user_data["segment_label"]
    else:
        age, segment = 25, "Unknown"

    bucket = "<25" if age < 25 else "25-35" if age <= 35 else "36-50" if age <= 50 else ">50"
    trending = cold_start_dict.get(bucket, [])
    
    if not current_cart:
        return trending[:top_k]

    cart_set = set(current_cart)

    # NEW FIX: Fast rule matching using precomputed Python sets (Bypasses slow Pandas loops)
    candidate_idxs = set()
    for item in cart_set:
        candidate_idxs.update(item_to_rules.get(item, set()))
        
    valid_idxs = [i for i in candidate_idxs if antecedent_sets[i].issubset(cart_set)]
    matches = final_rules.iloc[valid_idxs].copy() if valid_idxs else pd.DataFrame()

    if matches.empty:
        return [item for item in trending if item not in cart_set][:top_k]

    # 3. VIP Scoring (ECR)
    if segment in ["Mature Champions", "Young Whales"]:
        # NEW FIX: ECR formula uses Normalized Price and retains Lift
        matches["vip_score"] = matches["lift"] * matches["confidence"] * matches["price_norm"]
        matches = matches.sort_values("vip_score", ascending=False)
    else:
        matches = matches.sort_values("lift", ascending=False)

    recs = []
    # Fill from Rules
    for item in matches["consequent_item"].tolist():
        if item not in cart_set and item not in recs:
            recs.append(item)
        if len(recs) >= top_k:
            break

    # Pad from Trending
    if len(recs) < top_k:
        for item in trending:
            if item not in cart_set and item not in recs:
                recs.append(item)
            if len(recs) >= top_k:
                break
    return recs

# --- 10. DEMONSTRATION (TESTING THE ENGINE) ---
print("\n--- TEST 1: Cold-Start User (Empty Cart, Default Age <25) ---")
print("Recommended IDs:", baseline_recommendation(customer_id=999999999, current_cart=None))

if not final_rules.empty:
    sample_cart = final_rules.iloc[0]["antecedents_list"]
    print(f"\n--- TEST 2: Existing User matching items in cart {sample_cart} ---")
    print("Recommended IDs:", baseline_recommendation(customer_id=999999999, current_cart=sample_cart))


--- TEST 1: Cold-Start User (Empty Cart, Default Age <25) ---
Recommended IDs: [87467, 57656, 83608, 57643, 104045, 60763, 3711, 95499, 81746, 103479, 13713, 57645]

--- TEST 2: Existing User matching items in cart [88558] ---
Recommended IDs: [88544, 87467, 57656, 83608, 57643, 104045, 60763, 3711, 95499, 81746, 103479, 13713]


### **8. Offline Evaluation (MAP@12 & Hit Rate@12)**
Evaluating the baseline on the hold-out test set to establish the benchmark metrics. This serves to measure the lift provided by the subsequent personalized ALS and Deep Learning models.

In [ ]:
import time
import numpy as np

print('=' * 60)
print('Step 8: Offline Evaluation (Baseline)')
print('=' * 60)
t0 = time.time()

TEST_PATH = DATA_DIR / "test_processed.parquet"

# NEW FIX: Load Ground Truth and forcefully cast customer_id to integer for safe lookups
gt_df = (
    pl.scan_parquet(TEST_PATH)
    .group_by('customer_id')
    .agg(pl.col('article_id').unique().alias('actual'))
    .collect()
)
ground_truth = {int(r['customer_id']): set(r['actual']) for r in gt_df.iter_rows(named=True)}
print(f'  Ground truth: {len(ground_truth):,} users')

def ap_at_k(predicted, actual, k=12):
    """Average Precision at K."""
    if not actual: return 0.0
    hits, total = 0, 0.0
    for i, p in enumerate(predicted[:k]):
        if p in actual:
            hits += 1
            total += hits / (i + 1)
    return total / min(len(actual), k)

# NEW FIX: Extract ONLY the Last Known Basket (Max Date) for true RecSys Evaluation
print("Extracting Last Known Baskets for Evaluation...")

last_dates = (
    pl.scan_parquet(TRAIN_PATH)
    .group_by("customer_id")
    .agg(pl.col("t_dat").max().alias("last_date"))
)

last_baskets = (
    pl.scan_parquet(TRAIN_PATH)
    .join(last_dates, on="customer_id", how="inner")
    .filter(pl.col("t_dat") == pl.col("last_date"))
    .group_by("customer_id")
    .agg(pl.col("article_id").unique().alias("last_basket"))
    .collect()
)

# NEW FIX: Dictionary mapping user -> last basket (Cast to integer to match ground truth)
train_history_dict = {int(r["customer_id"]): r["last_basket"] for r in last_baskets.iter_rows(named=True)}

ap_scores = []
hits, evaluated = 0, 0
total_users = len(ground_truth)
print(f"Evaluating Baseline on {total_users} users...")

for i, (uid, actual_set) in enumerate(ground_truth.items()):
    
    # Pass ONLY the user's last known basket to the engine
    hist_cart = train_history_dict.get(uid, [])
    pred = baseline_recommendation(customer_id=uid, current_cart=hist_cart)
    
    ap = ap_at_k(pred, actual_set, k=12)
    ap_scores.append(ap)
    
    if actual_set & set(pred[:12]):
        hits += 1
    evaluated += 1
    
    if (i + 1) % 50000 == 0:
        print(f" Processed {i + 1:,} users...")

map12 = np.mean(ap_scores) if ap_scores else 0.0
hr12 = hits / evaluated if evaluated else 0.0

print(f'\n Evaluated   : {evaluated:,}')
print(f' MAP@12      : {map12:.6f}')
print(f' Hit Rate@12 : {hr12:.4f} ({hits:,}/{evaluated:,})')
print(f' Evaluation done in {time.time()-t0:.1f}s')

Step 8: Offline Evaluation (Baseline)
  Ground truth: 216,479 users
Extracting Last Known Baskets for Evaluation...
Evaluating Baseline on 216479 users...
 Processed 50,000 users...
 Processed 100,000 users...
 Processed 150,000 users...
 Processed 200,000 users...

 Evaluated   : 216,479
 MAP@12      : 0.004517
 Hit Rate@12 : 0.0444 (9,606/216,479)
 Evaluation done in 119.2s
